# EDA — Multimodal clustering (Project 2)

This notebook performs exploratory data analysis for the multimodal clustering pipeline.

Objectives:
- List and sanity-check available data files.
- Inspect schema, missing values, and temporal distributions.
- Basic text / numeric feature summaries.
- Quick embedding diagnostics and baseline clustering sample.
- Save a short EDA report to `report/`.

In [ ]:
# Basic environment check and file listing
import sys, os
print("Python:", sys.version)
print("CWD:", os.getcwd())
DATA_DIR = "data"
print("Data dir exists:", os.path.exists(DATA_DIR))
if os.path.exists(DATA_DIR):
    print("Files:", sorted(os.listdir(DATA_DIR))[:80])

In [ ]:
# Try imports and show versions
import importlib

def try_import(name):
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, "__version__", None)
        print(f"{name}: OK, version={ver}")
        return mod
    except Exception as e:
        print(f"{name}: ERROR ({e})")
        return None

pd = try_import("pandas")
np = try_import("numpy")
mpl = try_import("matplotlib")
if mpl:
    try:
        plt = importlib.import_module("matplotlib.pyplot")
        print("matplotlib.pyplot: OK")
    except Exception as e:
        print("matplotlib.pyplot: ERROR", e)

sns = try_import("seaborn")
sklearn = try_import("sklearn")
try:
    from sklearn.decomposition import PCA
    from sklearn.metrics import silhouette_score
    print("sklearn submodules: OK")
except Exception as e:
    print("sklearn submodules error:", e)

umap = try_import("umap")
hdbscan = try_import("hdbscan")

print('\nIf imports failed, install requirements with:')
print('pip install -r requirements.txt')

In [ ]:
# Load sample rows for quick EDA
if pd is None:
    raise SystemExit("pandas is required for EDA")

news_path = os.path.join(DATA_DIR, "news_clean.csv")
features_path = os.path.join(DATA_DIR, "features_aggregated.csv")
emb_path = os.path.join(DATA_DIR, "embeddings.npy")
emb_meta_path = os.path.join(DATA_DIR, "embeddings_meta.csv")

for k,v in dict(news=news_path, features=features_path, emb=emb_path, emb_meta=emb_meta_path).items():
    print(k, v, os.path.exists(v))

sample_n = 2000
if os.path.exists(news_path):
    df_news = pd.read_csv(news_path, nrows=sample_n)
    display(df_news.head())
else:
    print("No news file found.")

if os.path.exists(features_path):
    df_features = pd.read_csv(features_path, nrows=sample_n)
    display(df_features.head())
else:
    print("No features file found.")

In [ ]:
# Schema and missing values
for name, df in [("news", globals().get("df_news")), ("features", globals().get("df_features"))]:
    if df is None:
        continue
    print(f"--- {name} shape:", df.shape)
    print(df.dtypes)
    print("Missing rate per column:")
    print(df.isna().mean().sort_values(ascending=False).head(20))
    print("Unique counts:")
    print(df.nunique().sort_values(ascending=False).head(20))
    print()

In [ ]:
# Temporal checks
if 'df_news' in globals() and 'timestamp' in df_news.columns:
    df_news['ts'] = pd.to_datetime(df_news['timestamp'], errors='coerce')
    print("Time range:", df_news['ts'].min(), "-", df_news['ts'].max())
    counts = df_news.set_index('ts').resample('D').size()
    display(counts.head())
    try:
        import matplotlib.pyplot as plt
        counts.plot(title="Daily counts (sample)")
        plt.show()
    except Exception as e:
        print("Plot failed:", e)
else:
    print("No 'timestamp' column detected in news sample.")

In [ ]:
# Basic text EDA
text_col = None
for c in ['headline', 'title', 'text', 'content']:
    if 'df_news' in globals() and c in df_news.columns:
        text_col = c
        break

if text_col:
    s = df_news[text_col].dropna().astype(str)
    print("Text column:", text_col)
    s_len = s.str.len()
    print("Length stats:", s_len.describe())
    try:
        import matplotlib.pyplot as plt
        if 'seaborn' in globals() and globals().get('seaborn') is not None:
            import seaborn as sns
            sns.histplot(s_len, bins=50)
            plt.title("Text length distribution")
            plt.show()
    except Exception as e:
        print("Plotting text length failed:", e)
    from collections import Counter
    import re
    tokens = Counter()
    for t in s.sample(min(500, len(s))):
        tokens.update(re.findall(r"\w+", t.lower()))
    print("Top tokens:", tokens.most_common(30))
else:
    print("No text-like column found in news sample.")

In [ ]:
# Numeric features overview
if 'df_features' in globals():
    num = df_features.select_dtypes(include=['number'])
    print("Numeric columns:", num.columns.tolist())
    display(num.describe().T)
    if num.shape[1] > 1:
        try:
            import seaborn as sns
            import matplotlib.pyplot as plt
            corr = num.corr()
            plt.figure(figsize=(8,6))
            sns.heatmap(corr, vmin=-1, vmax=1, cmap='coolwarm')
            plt.title("Numeric feature correlations")
            plt.show()
        except Exception as e:
            print("Correlation plot failed:", e)
else:
    print("No features dataset loaded.")

In [ ]:
# Embeddings checks
if os.path.exists(emb_path):
    try:
        emb = np.load(emb_path, mmap_mode='r')
        print("Embeddings shape:", emb.shape)
        norms = (emb**2).sum(axis=1)**0.5
        print("Emb norms stats:", norms.min(), norms.mean(), norms.max())
        try:
            from sklearn.decomposition import PCA
            X2 = PCA(2).fit_transform(emb[:5000] if emb.shape[0]>5000 else emb)
            import matplotlib.pyplot as plt
            plt.figure(figsize=(6,5))
            plt.scatter(X2[:,0], X2[:,1], s=5, alpha=0.6)
            plt.title("PCA(2) of embeddings (sample)")
            plt.show()
        except Exception as e:
            print("Embed visualization failed:", e)
    except Exception as e:
        print("Could not load embeddings:", e)
else:
    print("No embeddings file found.")

In [ ]:
# Baseline clustering (small sample)
if 'emb' in globals():
    try:
        from umap import UMAP
        reducer = UMAP(n_components=2, random_state=42)
        X2 = reducer.fit_transform(emb[:5000] if emb.shape[0]>5000 else emb)
        try:
            import hdbscan
            labels = hdbscan.HDBSCAN(min_cluster_size=50).fit_predict(X2)
            print("HDBSCAN labels stats:", pd.Series(labels).value_counts().head())
            if (labels!=-1).sum() > 10:
                from sklearn.metrics import silhouette_score
                print("Silhouette (excluding noise):", silhouette_score(X2[labels!=-1], labels[labels!=-1]))
        except Exception as e:
            print("HDBSCAN not available:", e)
            from sklearn.cluster import KMeans
            km = KMeans(n_clusters=10, random_state=42)
            lbls = km.fit_predict(X2)
            print("KMeans cluster sizes:", pd.Series(lbls).value_counts().head())
            print("Silhouette:", silhouette_score(X2, lbls))
    except Exception as e:
        print("Clustering failed:", e)
else:
    print("No embeddings variable to cluster.")

In [ ]:
# Save a short summary file
out = "report/eda_summary.txt"
os.makedirs("report", exist_ok=True)
with open(out, "w", encoding="utf8") as f:
    f.write("EDA notebook run — check outputs and visualizations in this notebook.\n")
print("Wrote", out)